### 🧪 核心逻辑解释：因果特征提取 (Elasticity & Rigidity)

本单元格将原始交易数据转化为**因果特征**，用于识别非人类操作的洗钱脚本。

#### 1. 路径安全管理 (Path Safety)
* **逻辑**：使用 `pathlib` 取代硬编码的字符串（如 `../data`）。
* **优势**：它能自动识别项目根目录。无论你在 `research/` 还是 `scripts/` 文件夹下运行，代码都能精准找到数据文件，不会报错。

#### 2. 价格弹性公式 ($\beta$)
我们使用对数线性回归逻辑来计算**价格弹性**。对于每个地址，我们计算其交易量对 Gas 价格变化的响应：
$$\beta = \frac{\text{Cov}(\ln Q, \ln P)}{\text{Var}(\ln P)}$$
* **$\ln Q$ (`log_q`)**：USDC 赎回金额的自然对数（加 `1e-9` 防止对 0 取对数）。
* **$\ln P$ (`log_p`)**：Gas 价格 (Gwei) 的自然对数。
* **因果含义**：
    * **$\beta < -1$**：**高弹性**。典型的理性套利者，Gas 贵了就少转账，Gas 便宜了就多转。
    * **$\beta \approx 0$**：**成本刚性 (Rigidity)**。极度可疑，说明其行为完全不受手续费波动影响，通常是脚本在执行强制指令。

#### 3. 特征工程：刚性得分 (Rigidity Score)
* **`rigidity_score`**：我们对弹性取绝对值 $|elasticity|$。
* **排序意义**：数值越接近 **0**，说明该地址的行为越“冷酷”。我们将得分最低（最刚性）的地址排在最前面作为主要嫌疑对象。

#### 4. Polars 链式调用优化
* **链式操作**：通过 `( ... )` 将所有步骤串联。
* **处理流程**：增加对数列 -> 按地址聚合 -> 过滤掉交易次数少于 2 次的地址（无法计算方差）-> 计算弹性 -> 排序。

Path Safety: I used Pathlib instead of ../. This ensures the code finds your data even if you move the notebook between folders.

Rigidity Score: I added a new column called rigidity_score. In Causal ML, we are looking for values near zero. By taking the absolute value $|elasticity|$, you can easily sort and find the most "frozen" (rigid) users at the top.

Method Chaining: I wrapped the logic in parentheses ( ... ). This is a "clean code" practice that makes the steps (Group -> Agg -> Filter -> Sort) look like a readable list.

Formula Accuracy: Your logic correctly implements the elasticity coefficient $\beta$ using the OLS (Ordinary Least Squares) approach:

In [18]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 1. Setup Robust Paths (Works regardless of where your notebook is)
root = Path.cwd().parent if Path.cwd().name in ["research", "scripts"] else Path.cwd()
data_file = root / "data" / "raw" / "usdc_burns_48h.parquet"

# 2. Load and Transform
df = pl.read_parquet(data_file)

# Calculate Elasticity and add a "Rigidity Score"
# Elasticity formula: Beta = Cov(log_q, log_p) / Var(log_p)
entity_metrics = (
    df.with_columns([
        (pl.col("amount_usdc") + 1e-9).log().alias("log_q"),
        (pl.col("gas_price_gwei") + 1e-9).log().alias("log_p")
    ])
    .group_by("user_address")
    .agg([
        pl.count("evt_tx_hash").alias("tx_count"),
        pl.col("amount_usdc").mean().alias("avg_amount"),
        ((pl.cov("log_q", "log_p")) / (pl.col("log_p").var())).alias("elasticity")
    ])
    .filter(pl.col("tx_count") >= 2)
    .drop_nulls()
    # Add Rigidity Score: Absolute value closer to 0 means MORE rigid (suspicious)
    .with_columns(
        pl.col("elasticity").abs().alias("rigidity_score")
    )
    .sort("rigidity_score")
)

# 3. Quick Visual Summary
print(f"✅ Analysis Complete. Total unique entities: {entity_metrics.height}")
entity_metrics.head(10)

✅ Analysis Complete. Total unique entities: 2


user_address,tx_count,avg_amount,elasticity,rigidity_score
str,u32,f64,f64,f64
"""0x55fe002aeff02f77364de339a129…",96,4.0573e6,-0.340566,0.340566
"""0xc4922d64a24675e16e1586e3e3aa…",1360,57324.26847,0.496664,0.496664


In [20]:
import polars as pl
from pathlib import Path

# 1. 自动定位数据
root = Path.cwd().parent if Path.cwd().name in ["research", "scripts"] else Path.cwd()
data_file = root / "data" / "raw" / "usdc_burns_48h.parquet"

# 2. 加载数据
df = pl.read_parquet(data_file)

# 3. 寻找 Gas 最高峰时刻 (The Stress Peak)
max_gas_row = df.filter(pl.col("gas_price_gwei") == pl.col("gas_price_gwei").max())
peak_time = max_gas_row["evt_block_time"][0]
max_gas_val = max_gas_row["gas_price_gwei"][0]

print(f"📊 检测到全段最高 Gas 压力点: {max_gas_val:.2f} Gwei")
print(f"⏰ 压力高峰时刻: {peak_time}")

# 4. 定义“高压窗口” (高峰前后各 30 分钟，共 1 小时)
# 这是因果审计的核心：观察极端环境下的行为
buffer = 30 # 分钟
start_win = peak_time - pl.duration(minutes=buffer)
end_win = peak_time + pl.duration(minutes=buffer)

# 5. 提取窗口内的 Top 10 活跃地址 (嫌疑名单)
suspect_list = (
    df.filter((pl.col("evt_block_time") >= start_win) & (pl.col("evt_block_time") <= end_win))
    .group_by("user_address")
    .agg([
        pl.count("evt_tx_hash").alias("tx_count"),
        pl.col("amount_usdc").sum().alias("total_burn_volume"),
        pl.col("gas_price_gwei").mean().alias("avg_window_gas")
    ])
    .sort("total_burn_volume", descending=True)
    .head(10)
)

print(f"🚨 在 Gas 峰值前后 {buffer}min 内，交易额最大的 Top 10 地址：")
suspect_list

📊 检测到全段最高 Gas 压力点: 663.73 Gwei
⏰ 压力高峰时刻: 2024-08-05 01:15:11+00:00
🚨 在 Gas 峰值前后 30min 内，交易额最大的 Top 10 地址：


user_address,tx_count,total_burn_volume,avg_window_gas
str,u32,f64,f64
"""0x55fe002aeff02f77364de339a129…",2,1.1629e7,337.504656
"""0xc4922d64a24675e16e1586e3e3aa…",23,1.5045e6,238.074287


In [21]:
import os
import polars as pl
import matplotlib.pyplot as plt

# 1. 自动定位当前目录，确保文件夹存在
base_dir = os.getcwd()
data_dir = os.path.join(base_dir, "data", "raw")
results_dir = os.path.join(base_dir, "results")

os.makedirs(data_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print(f"📂 目标存储路径已确认:\n数据: {data_dir}\n图表: {results_dir}")

# 2. 检查变量 df 是否还在内存中（假设你刚才跑过 fetch 脚本）
# 如果不在，请重新运行一次 fetch_and_process() 里的逻辑
if 'raw_rows' in locals() or 'raw_rows' in globals():
    # 重新构建并清洗
    df = pl.DataFrame(raw_rows).with_columns([
        pl.col("amount_usdc").cast(pl.Float64),
        pl.col("gas_price_gwei").cast(pl.Float64),
        pl.col("evt_block_time").str.to_datetime(format="%Y-%m-%d %H:%M:%S%.3f UTC").dt.replace_time_zone("UTC")
    ]).sort("evt_block_time")

    # 3. 强制写入
    parquet_file = os.path.join(data_dir, "usdc_burns_48h.parquet")
    df.write_parquet(parquet_file)
    print(f"✅ 数据文件已生成: {parquet_file}")

    # 4. 快速生成图表预览
    df_resampled = df.group_by_dynamic("evt_block_time", every="5m").agg([
        pl.col("amount_usdc").sum().alias("vol"),
        pl.col("gas_price_gwei").mean().alias("gas")
    ])

    plt.figure(figsize=(10, 4))
    plt.plot(df_resampled["evt_block_time"], df_resampled["gas"], color='red', label='Gas Price')
    plt.title("Gas Price Fluctuation (Demo Check)")
    
    plot_file = os.path.join(results_dir, "gas_check.png")
    plt.savefig(plot_file)
    plt.show()
    print(f"🖼️ 图表预览已生成: {plot_file}")
else:
    print("❌ 内存中找不到 raw_rows，请先运行 fetch 脚本的代码块获取数据！")

📂 目标存储路径已确认:
数据: c:\Users\AAA\Desktop\项目\Gas-Stress-Test-AML\scripts\data\raw
图表: c:\Users\AAA\Desktop\项目\Gas-Stress-Test-AML\scripts\results
❌ 内存中找不到 raw_rows，请先运行 fetch 脚本的代码块获取数据！
